# Hate Speech Detection — Full Pipeline Demo

**Architecture (no routing):**
```
text → [Layer 2] retrieve neighbors → augment → RAG classifier → label + confidence
     → [Layer 3] LLM explanation → structured moderation output
```

Evaluated on 50 real 4chan posts with ground-truth labels (`hatespeech_dataset_4chan.xlsx`).  
Edit **Cell 3** to switch model configuration.

In [4]:
import sys, os, json, re, torch, faiss
import torch.nn.functional as F
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, classification_report, confusion_matrix
)
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

sys.path.insert(0, str(Path(".").resolve()))  # makes rag.py and layer3_explainer.py importable
from rag import retrieve_top_k, retrieve_top_k_above_threshold
from layer3_explainer import explain, Layer2Output

/Users/alexandre/anaconda3/envs/dl_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Cell 2 — LLM Backend

In [5]:
# --- Option A: Groq (free, recommended) — get a key at console.groq.com ---
from groq import Groq
llm_client = Groq(api_key="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")
LLM_MODEL = "llama-3.3-70b-versatile"

# --- Option B: Ollama (local, free) ---
# import openai
# llm_client = openai.OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# LLM_MODEL = "mistral"

# --- Option C: OpenAI (paid) ---
# import openai
# llm_client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY", ""))
# LLM_MODEL = "gpt-4o-mini"

print(f"LLM backend ready: {LLM_MODEL}")

LLM backend ready: llama-3.3-70b-versatile


## Cell 3 — Pipeline Config
`INDEX_SPLIT` controls both which FAISS index is queried and which classifier weights are loaded — they must match.

In [ ]:
MODEL_FAMILY = "roberta"   # "bert" | "hatebert" | "roberta"
INDEX_SPLIT  = "documents"  # "training" | "documents" | "full"
DATASET      = "ISHate"    # "IHC" | "ISHate" | "Vicomtech"
K            = 5           # max neighbors to retrieve
THRESHOLD    = 0.98        # min cosine similarity; set to 0.0 to always get K neighbors

## Cell 4 — Load Dataset

In [7]:
df = pd.read_csv("../hatespeech_dataset_4chan.csv", sep=None, engine="python")
print(f"Dataset: {len(df)} examples")
print(f"Label distribution: {df['label'].value_counts().to_dict()}")
df.head()

Dataset: 50 examples
Label distribution: {'not hate': 32, 'hate': 18}


,id,text,label,Unnamed: 3,Unnamed: 4
0,1,Bitches are dumb yes but he was busted on grin...,hate,NaN,NaN
1,2,Satanic trannies are skipping rehearsal?,hate,NaN,NaN
2,3,go back to Africa nigger,hate,NaN,NaN
3,4,The Jewish people are very good people. So are...,not hate,NaN,NaN
4,5,Oh BTW this was the first year where phones be...,not hate,NaN,NaN


## Cell 5 — Load Pipeline Components

In [8]:
HF_IDS = {
    "bert":     "bert-base-uncased",
    "hatebert": "GroNLP/hateBERT",
    "roberta":  "roberta-base",
}

def load_pipeline(model_family, index_split, dataset):
    hf_id  = HF_IDS[model_family]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    print(f"Loading retriever: {hf_id} ...")
    ret_tokenizer = AutoTokenizer.from_pretrained(hf_id)
    ret_model     = AutoModel.from_pretrained(hf_id).eval().to(device)

    index_path  = f"index/{model_family}/base/vdb_{index_split}.faiss"
    lookup_path = f"index/lookup_{index_split}.json"
    print(f"Loading index: {index_path} ...")
    index = faiss.read_index(index_path)
    with open(lookup_path) as f:
        documents = json.load(f)
    print(f"  Index size: {index.ntotal:,} vectors")

    clf_path = f"../weights_rag/{model_family}/base/{index_split}/{dataset}"
    print(f"Loading RAG classifier: {clf_path} ...")
    clf_tokenizer = AutoTokenizer.from_pretrained(clf_path)
    clf_model     = AutoModelForSequenceClassification.from_pretrained(clf_path).eval().to(device)

    print(f"\nReady: {model_family.upper()} | index={index_split} | trained_on={dataset}")
    return ret_model, ret_tokenizer, index, documents, clf_model, clf_tokenizer, device


ret_model, ret_tokenizer, index, documents, clf_model, clf_tokenizer, device = load_pipeline(
    MODEL_FAMILY, INDEX_SPLIT, DATASET
)

Device: cpu
Loading retriever: roberta-base ...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 16536.47it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading index: index/roberta/base/vdb_documents.faiss ...
  Index size: 40,950 vectors
Loading RAG classifier: ../weights_rag/roberta/base/documents/ISHate ...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 17986.33it/s]


Ready: ROBERTA | index=documents | trained_on=ISHate


## Cell 6 — Pipeline Runner and Display

In [9]:
_LABEL_RE = re.compile(r"^\[(hate|not hate)\]\s*:?\s*", re.IGNORECASE)

def strip_label(text):
    return _LABEL_RE.sub("", text).strip()


def run_pipeline(text, ret_model, ret_tokenizer, index, documents,
                 clf_model, clf_tokenizer, device,
                 llm_client, llm_model, k=K, threshold=THRESHOLD):

    # Layer 2a: retrieve neighbors
    retrieved = retrieve_top_k_above_threshold(
        text, threshold, ret_model, ret_tokenizer, index, documents, chunk_id=None, k=k
    )
    if not retrieved:  # fallback: nothing cleared the threshold
        retrieved = retrieve_top_k(
            text, ret_model, ret_tokenizer, index, documents, chunk_id=None, k=k
        )

    # Layer 2b: augment and classify
    sep = clf_tokenizer.sep_token or "[SEP]"
    augmented = f" {sep} ".join([text] + [t for t, _ in retrieved])
    inputs = clf_tokenizer(
        augmented, return_tensors="pt", truncation=True, padding=True, max_length=256
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = clf_model(**inputs).logits[0]
    probs      = F.softmax(logits, dim=-1)
    pred_idx   = torch.argmax(probs).item()
    label      = "hate" if pred_idx == 1 else "not hate"
    confidence = probs[pred_idx].item()

    # Layer 3: LLM explanation
    l2 = Layer2Output(
        original_text=text, label=label, confidence=confidence,
        hate_category="unknown", retrieved=retrieved
    )
    explanation = explain(l2, llm_client, llm_model)
    return l2, explanation


def display_result(example_id, text, ground_truth, l2, explanation):
    w = 80
    correct  = l2.label == ground_truth
    mark     = "\u2713" if correct else "\u2717"
    print("=" * w)
    print(f"[{example_id}] {mark}  TEXT : {text}")
    print(f"       GROUND TRUTH : {ground_truth.upper()}")
    print("-" * w)
    print(f"LAYER 2  : {l2.label.upper()}  ({l2.confidence:.1%} confidence)")
    print()
    print(f"RETRIEVED NEIGHBORS ({len(l2.retrieved)}):")
    for i, (txt, score) in enumerate(l2.retrieved, 1):
        print(f"  [{i}] {score:.4f}  {strip_label(txt)[:100]}")
    print()
    print("LAYER 3 EXPLANATION:")
    print(f"  Summary   : {explanation.summary}")
    print(f"  Severity  : {explanation.severity}")
    print(f"  Action    : {explanation.recommended_action}")
    print(f"  Targets   : {', '.join(explanation.target_groups) if explanation.target_groups else chr(8212)}")
    print(f"  Evidence  : {explanation.evidence_used}")
    if explanation.moderator_note:
        print(f"  Note      : {explanation.moderator_note}")
    valid_str = "\u2713 passed" if explanation.validation_passed else "\u2717 FAILED (forced human-review)"
    print(f"  Validation: {valid_str}")
    print("=" * w)
    print()

## Cell 7 — Run Full Pipeline on All 50 Examples

In [8]:
## Diagnostic — isolate crash point (run before Cell 7)
import faulthandler, numpy as np
faulthandler.enable()   # prints C-level traceback on segfault instead of silent crash

print("Step 1: encode one text...", end=" ", flush=True)
from rag import encode
_test_vec = encode(["test sentence"], ret_model, ret_tokenizer, batch_size=1)
print(f"OK  shape={_test_vec.shape}  dtype={_test_vec.dtype}")

print("Step 2: faiss.normalize_L2...", end=" ", flush=True)
_v = _test_vec.copy()
faiss.normalize_L2(_v)
print("OK")

print("Step 3: index.search...", end=" ", flush=True)
_scores, _ids = index.search(_v, 3)
print(f"OK  scores={_scores[0]}")

print("Step 4: numpy-based cosine sim (FAISS bypass)...", end=" ", flush=True)
# Extract stored vectors from Flat index for numpy fallback
_xb = faiss.vector_to_array(index.xb).reshape(index.ntotal, index.d)
_sims = (_xb @ _v.T).squeeze()
_top = np.argsort(_sims)[::-1][:3]
print(f"OK  top sims={_sims[_top].round(4)}")

print("\nAll steps passed — crash is NOT in basic FAISS ops on this run.")
print("If any step above crashes the kernel, that is the culprit.")

Step 1: encode one text... 

Encoding: 100%|██████████| 1/1 [00:00<00:00,  1.44it/s]

OK  shape=(1, 768)  dtype=float32
Step 2: faiss.normalize_L2... OK
Step 3: index.search... 

: 

In [10]:
import psutil, time, gc, numpy as np
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout
from layer3_explainer import ExplainerOutput
from rag import encode

print(f"RAM available: {psutil.virtual_memory().available / 1e9:.1f} GB")
print(f"Config : {MODEL_FAMILY.upper()} | index={INDEX_SPLIT} | trained_on={DATASET}")
print(f"Dataset: 4chan ({len(df)} examples)\n")

LLM_TIMEOUT      = 30  # seconds per Groq call before giving up
RATE_LIMIT_SLEEP = 2   # seconds between calls (Groq free tier: 30 req/min)

# Unwrap IndexIDMap → extract raw vectors + ID map for pure-numpy cosine search
print("Extracting index vectors for numpy search...", end=" ", flush=True)
_inner  = faiss.downcast_index(index.index)
_xb     = np.empty((index.ntotal, index.d), dtype="float32")
_inner.reconstruct_n(0, index.ntotal, _xb)                 # works on any Flat index version
_id_map = faiss.vector_to_array(index.id_map).astype("int64")
print(f"done  shape={_xb.shape}")

def retrieve_numpy(text, threshold, k, model, tokenizer):
    """Encode once, cosine sim via numpy — no FAISS at query time."""
    vec   = encode([text], model, tokenizer, batch_size=1)
    vec_n = vec / np.maximum(np.linalg.norm(vec, axis=1, keepdims=True), 1e-9)

    sims    = (_xb @ vec_n.T).squeeze()
    top_pos = np.argsort(sims)[::-1][:k]
    top_ids = _id_map[top_pos]
    scores  = sims[top_pos]

    retrieved = [
        (documents[str(int(cid))], float(sc))
        for cid, sc in zip(top_ids, scores)
        if sc >= threshold
    ][:k]
    if not retrieved:
        retrieved = [
            (documents[str(int(cid))], float(sc))
            for cid, sc in zip(top_ids, scores)
        ][:k]

    del vec, vec_n, sims, top_pos, top_ids, scores
    return retrieved

records = []
t_start = time.time()

for i, (_, row) in enumerate(df.iterrows()):
    text         = str(row["text"])
    ground_truth = str(row["label"]).strip().lower()
    example_id   = int(row["id"])

    t0  = time.time()
    ram = psutil.virtual_memory().available / 1e9
    print(f"[{i+1:02d}/50] id={example_id}  RAM={ram:.1f}GB  ...", end=" ", flush=True)

    retrieved = retrieve_numpy(text, THRESHOLD, K, ret_model, ret_tokenizer)

    sep       = clf_tokenizer.sep_token or "[SEP]"
    augmented = f" {sep} ".join([text] + [t for t, _ in retrieved])
    inputs    = clf_tokenizer(augmented, return_tensors="pt", truncation=True, padding=True, max_length=256)
    inputs    = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = clf_model(**inputs).logits[0]
    del inputs
    probs      = torch.nn.functional.softmax(logits, dim=-1)
    pred_idx   = torch.argmax(probs).item()
    label      = "hate" if pred_idx == 1 else "not hate"
    confidence = probs[pred_idx].item()
    del logits, probs

    l2 = Layer2Output(
        original_text=text, label=label, confidence=confidence,
        hate_category="unknown", retrieved=retrieved,
    )

    try:
        with ThreadPoolExecutor(max_workers=1) as ex:
            fut         = ex.submit(explain, l2, llm_client, LLM_MODEL)
            explanation = fut.result(timeout=LLM_TIMEOUT)
    except FuturesTimeout:
        print(f"TIMEOUT({LLM_TIMEOUT}s) ", end="", flush=True)
        explanation = ExplainerOutput(
            summary="LLM call timed out.", evidence_used=[], target_groups=[],
            severity="unknown", recommended_action="human-review",
            moderator_note="Groq call exceeded timeout.", validation_passed=False,
        )
    except Exception as e:
        print(f"ERR({e}) ", end="", flush=True)
        explanation = ExplainerOutput(
            summary=f"LLM error: {e}", evidence_used=[], target_groups=[],
            severity="unknown", recommended_action="human-review",
            moderator_note="LLM call raised an exception.", validation_passed=False,
        )

    elapsed = time.time() - t0
    correct = label == ground_truth
    print(f"{'✓' if correct else '✗'}  gt={ground_truth.upper():8s} pred={label.upper():8s}  conf={confidence:.1%}  {elapsed:.1f}s")

    records.append({
        "id"               : example_id,
        "text"             : text,
        "ground_truth"     : ground_truth,
        "predicted"        : label,
        "confidence"       : round(confidence, 4),
        "n_retrieved"      : len(retrieved),
        "top_sim"          : round(retrieved[0][1], 4) if retrieved else None,
        "evidence_used"    : explanation.evidence_used,
        "severity"         : explanation.severity,
        "action"           : explanation.recommended_action,
        "target_groups"    : explanation.target_groups,
        "validation_passed": explanation.validation_passed,
        "correct"          : correct,
    })

    gc.collect()
    if i < len(df) - 1:
        time.sleep(RATE_LIMIT_SLEEP)

results_df = pd.DataFrame(records)
total = time.time() - t_start
print(f"\nDone. {results_df['correct'].sum()}/{len(results_df)} correct  |  total={total/60:.1f} min")

RAM available: 7.0 GB
Config : ROBERTA | index=documents | trained_on=ISHate
Dataset: 4chan (50 examples)

Extracting index vectors for numpy search... done  shape=(40950, 768)
[01/50] id=1  RAM=7.0GB  ... 

Encoding: 100%|██████████| 1/1 [00:00<00:00, 17.30it/s]


✗  gt=HATE     pred=NOT HATE  conf=98.7%  1.2s
[02/50] id=2  RAM=7.2GB  ... 

Encoding: 100%|██████████| 1/1 [00:00<00:00, 16.28it/s]


✓  gt=HATE     pred=HATE      conf=99.5%  1.6s
[03/50] id=3  RAM=7.0GB  ... 

Encoding: 100%|██████████| 1/1 [00:00<00:00, 18.08it/s]


✓  gt=HATE     pred=HATE      conf=100.0%  0.8s
[04/50] id=4  RAM=7.1GB  ... 

Encoding: 100%|██████████| 1/1 [00:00<00:00, 16.64it/s]


✗  gt=NOT HATE pred=HATE      conf=99.7%  1.1s


KeyboardInterrupt: 

## Cell 8 — Classification Metrics

In [19]:
y_true = results_df["ground_truth"].tolist()
y_pred = results_df["predicted"].tolist()
labels = ["hate", "not hate"]

print(f"Config : {MODEL_FAMILY.upper()} | index={INDEX_SPLIT} | trained_on={DATASET}")
print(f"Dataset: 4chan ({len(y_true)} examples)")
print("=" * 50)
print(f"Accuracy  : {accuracy_score(y_true, y_pred):.3f}")
print(f"F1 macro  : {f1_score(y_true, y_pred, average='macro'):.3f}")
print(f"Precision : {precision_score(y_true, y_pred, average='macro'):.3f}")
print(f"Recall    : {recall_score(y_true, y_pred, average='macro'):.3f}")
print()
print(classification_report(y_true, y_pred, target_names=labels))

cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_df = pd.DataFrame(
    cm,
    index=[f"True: {l}" for l in labels],
    columns=[f"Pred: {l}" for l in labels]
)
print("Confusion matrix:")
display(cm_df)

Config : ROBERTA | index=documents | trained_on=ISHate
Dataset: 4chan (50 examples)
Accuracy  : 0.640
F1 macro  : 0.625
Precision : 0.625
Recall    : 0.634

              precision    recall  f1-score   support

        hate       0.50      0.61      0.55        18
    not hate       0.75      0.66      0.70        32

    accuracy                           0.64        50
   macro avg       0.62      0.63      0.62        50
weighted avg       0.66      0.64      0.65        50

Confusion matrix:


,Pred: hate,Pred: not hate
True: hate,11,7
True: not hate,11,21


## Cell 9 — Results Summary Table

In [20]:
display(results_df[[
    "id", "ground_truth", "predicted", "confidence",
    "n_retrieved", "top_sim", "severity", "action",
    "validation_passed", "correct"
]])

,id,ground_truth,predicted,confidence,n_retrieved,top_sim,severity,action,validation_passed,correct
0,1,hate,not hate,0.9866,3,0.9974,unknown,human-review,False,False
1,2,hate,hate,0.9955,3,0.9979,unknown,human-review,False,True
2,3,hate,hate,0.9998,3,0.9985,unknown,human-review,False,True
3,4,not hate,hate,0.9970,3,0.9978,unknown,human-review,False,False
4,5,not hate,not hate,0.9990,3,0.9981,unknown,human-review,False,True
5,6,not hate,hate,0.9929,3,0.9979,unknown,human-review,False,False
6,7,not hate,not hate,0.9953,3,0.9973,unknown,human-review,False,True
7,8,not hate,not hate,0.9940,3,0.9969,unknown,human-review,False,True
8,9,not hate,not hate,0.9864,3,0.9971,unknown,human-review,False,True
9,10,not hate,not hate,0.9992,3,0.9977,unknown,human-review,False,True


## Cell 10 — Error Analysis

In [10]:
errors = results_df[~results_df["correct"]].reset_index(drop=True)
print(f"Misclassified: {len(errors)}/{len(results_df)}\n")
for _, row in errors.iterrows():
    print(f"[{int(row['id'])}] GT={row['ground_truth'].upper():8s}  PRED={row['predicted'].upper():8s}  conf={row['confidence']:.1%}")
    print(f"       {row['text'][:120]}")
    print()

Misclassified: 17/50

[1] GT=HATE      PRED=NOT HATE  conf=92.0%
       Bitches are dumb yes but he was busted on grindr or some other gay app you queer.

[4] GT=NOT HATE  PRED=HATE      conf=99.9%
       The Jewish people are very good people. So are the White people. Everyone should just simmer down and take a chill pill

[6] GT=NOT HATE  PRED=HATE      conf=96.6%
       IPAs are disgusting and like 7 dollars a pop. I hate going to bars now.

[13] GT=NOT HATE  PRED=HATE      conf=100.0%
       integral watches are so hideous

[14] GT=HATE      PRED=NOT HATE  conf=99.5%
       You will be drafted to defend zog

[16] GT=NOT HATE  PRED=HATE      conf=99.3%
       Lots of blacks use fentanyl as well. Meth is mostly a white thing though.

[18] GT=NOT HATE  PRED=HATE      conf=100.0%
       Incitement to violence is a crime.

[20] GT=HATE      PRED=NOT HATE  conf=98.8%
       Make sure they know it was Cohen slave boats

[26] GT=HATE      PRED=NOT HATE  conf=92.0%
       The genocide was a

## Cell 11 — Compare All 27 Configs *(optional — slow, ~45 min on CPU)*

Runs every `(model, index_split, dataset)` combination on the full 4chan dataset and ranks by F1.

In [16]:
## Cell 10b — Inspect raw LLM output for one example
from layer3_explainer import build_prompt, call_llm, validate_output, _REQUIRED_FIELDS

# Pick any row index to inspect (0 = first example)
ROW_IDX = 49

row          = df.iloc[ROW_IDX]
text         = str(row["text"])
ground_truth = str(row["label"]).strip().lower()

# Retrieve neighbors (same numpy path as Cell 7)
vec   = encode([text], ret_model, ret_tokenizer, batch_size=1)
vec_n = vec / np.maximum(np.linalg.norm(vec, axis=1, keepdims=True), 1e-9)
sims    = (_xb @ vec_n.T).squeeze()
top_pos = np.argsort(sims)[::-1][:K]
top_ids = _id_map[top_pos]
scores  = sims[top_pos]
retrieved = [(documents[str(int(c))], float(s)) for c, s in zip(top_ids, scores)][:K]

print(f"Text        : {text}")
print(f"Ground truth: {ground_truth}")
print(f"Retrieved   : {len(retrieved)} neighbors")
for i, (t, s) in enumerate(retrieved, 1):
    print(f"  [{i}] {s:.4f}  {t[:100]}")
print()

# Build and print the prompt sent to the LLM
prompt = build_prompt(text, "hate", 0.99, "unknown", retrieved)
print("=" * 60)
print("PROMPT SENT TO LLM:")
print("=" * 60)
print(prompt)
print()

# Call LLM and show raw response
print("=" * 60)
print("RAW LLM RESPONSE:")
print("=" * 60)
try:
    raw = call_llm(prompt, llm_client, LLM_MODEL)
    import json
    print(json.dumps(raw, indent=2))
    print()
    valid = validate_output(raw, len(retrieved))
    print(f"validation_passed: {valid}")
    if not valid:
        print("Validation failure reasons:")
        if not _REQUIRED_FIELDS.issubset(raw.keys()):
            print(f"  Missing fields: {_REQUIRED_FIELDS - raw.keys()}")
        if not isinstance(raw.get("evidence_used"), list) or not raw.get("evidence_used"):
            print(f"  evidence_used is empty or not a list: {raw.get('evidence_used')}")
        else:
            bad = [i for i in raw["evidence_used"] if not isinstance(i, int) or not (1 <= i <= len(retrieved))]
            if bad:
                print(f"  evidence_used indices out of range [1,{len(retrieved)}]: {bad}")
        if raw.get("severity") not in {"low", "medium", "high"}:
            print(f"  Invalid severity: {raw.get('severity')!r}")
        if raw.get("recommended_action") not in {"auto-block", "human-review", "allow"}:
            print(f"  Invalid recommended_action: {raw.get('recommended_action')!r}")
except Exception as e:
    print(f"LLM call failed: {e}")
    import traceback; traceback.print_exc()

Encoding: 100%|██████████| 1/1 [00:00<00:00, 14.16it/s]


Text        : wtf fuck these mudslimic terrorists for killing these white men
Ground truth: hate
Retrieved   : 3 neighbors
  [1] 0.9977  [hate] Alex Jones: how many lemmings you get out there on the street, begging to have their guns tak
  [2] 0.9976  [hate] Antisemitism: for their own benefit. antisemitism undergirds much of the far right, unifying 
  [3] 0.9975  [hate] Center for Security Policy: anti - islam conspiracy theories. trento once addressed a crowd i

PROMPT SENT TO LLM:
TEXT TO MODERATE:
"wtf fuck these mudslimic terrorists for killing these white men"

CLASSIFICATION DECISION:
  label     : hate
  confidence: 0.99
  category  : unknown

RETRIEVED EVIDENCE PASSAGES:
[1] "Alex Jones: how many lemmings you get out there on the street, begging to have their guns taken. we will not relinquish them! do you understand? ” — cnn"  (similarity: 0.9977)
[2] "Antisemitism: for their own benefit. antisemitism undergirds much of the far right, unifying adherents in their belief that j

In [12]:
import psutil, time, gc, numpy as np
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout
from layer3_explainer import ExplainerOutput
from rag import encode
import traceback

ALL_CONFIGS = [
    (model, split, dataset)
    for model   in ["roberta"]
    for split   in ["training", "documents", "full"]
    for dataset in ["IHC", "ISHate", "Vicomtech"]
]

def extract_numpy_index(faiss_index):
    """Unwrap IndexIDMap and return (xb, id_map) as numpy arrays — no index.search() called."""
    inner  = faiss.downcast_index(faiss_index.index)
    xb     = np.empty((faiss_index.ntotal, faiss_index.d), dtype="float32")
    inner.reconstruct_n(0, faiss_index.ntotal, xb)
    id_map = faiss.vector_to_array(faiss_index.id_map).astype("int64")
    return xb, id_map

def classify_numpy(text, xb, id_map, docs, r_model, r_tok, c_model, c_tok, dev, k, threshold):
    """Full Layer 2 pipeline using numpy cosine search — no FAISS at query time."""
    vec   = encode([text], r_model, r_tok, batch_size=1)
    vec_n = vec / np.maximum(np.linalg.norm(vec, axis=1, keepdims=True), 1e-9)
    sims    = (xb @ vec_n.T).squeeze()
    top_pos = np.argsort(sims)[::-1][:k]
    top_ids = id_map[top_pos]
    scores  = sims[top_pos]

    retrieved = [(docs[str(int(c))], float(s)) for c, s in zip(top_ids, scores) if s >= threshold][:k]
    if not retrieved:
        retrieved = [(docs[str(int(c))], float(s)) for c, s in zip(top_ids, scores)][:k]
    del vec, vec_n, sims, top_pos, top_ids, scores

    sep       = c_tok.sep_token or "[SEP]"
    augmented = f" {sep} ".join([text] + [t for t, _ in retrieved])
    inputs    = c_tok(augmented, return_tensors="pt", truncation=True, padding=True, max_length=256)
    inputs    = {k: v.to(dev) for k, v in inputs.items()}
    with torch.no_grad():
        logits = c_model(**inputs).logits[0]
    del inputs
    probs  = torch.nn.functional.softmax(logits, dim=-1)
    label  = "hate" if torch.argmax(probs).item() == 1 else "not hate"
    del logits, probs
    return label

summary_rows = []

for model_f, split, dset in ALL_CONFIGS:
    config_name = f"{model_f}/{split}/{dset}"
    print(f"Running {config_name} ...", end=" ", flush=True)
    try:
        r_model, r_tok, idx, docs, c_model, c_tok, dev = load_pipeline(model_f, split, dset)
        xb, id_map = extract_numpy_index(idx)

        preds, truths = [], []
        for _, row in df.iterrows():
            pred = classify_numpy(
                str(row["text"]), xb, id_map, docs,
                r_model, r_tok, c_model, c_tok, dev, K, THRESHOLD
            )
            preds.append(pred)
            truths.append(str(row["label"]).strip().lower())

        acc = accuracy_score(truths, preds)
        f1  = f1_score(truths, preds, average="macro")
        summary_rows.append({"Config": config_name, "Accuracy": round(acc, 3), "F1 macro": round(f1, 3)})
        print(f"acc={acc:.3f}  f1={f1:.3f}")

        del r_model, c_model, xb, id_map
        gc.collect()
    except Exception as e:
        print(f"ERROR: {e}")
        traceback.print_exc()

if summary_rows:
    summary_df = pd.DataFrame(summary_rows).sort_values("F1 macro", ascending=False)
    display(summary_df)
else:
    print("\nNo configs completed — all raised errors (see above).")

Running roberta/training/IHC ... Device: cpu
Loading retriever: roberta-base ...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 18181.53it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading index: index/roberta/base/vdb_training.faiss ...
  Index size: 67,864 vectors
Loading RAG classifier: ../weights_rag/roberta/base/training/IHC ...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8655.95it/s]



Ready: ROBERTA | index=training | trained_on=IHC


...

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7364.47it/s]



Ready: ROBERTA | index=full | trained_on=Vicomtech


Encoding: 100%|██████████| 1/1 [00:00<00:00, 10.70it/s]


acc=0.760  f1=0.745


,Config,Accuracy,F1 macro
5,roberta/documents/Vicomtech,0.82,0.790
8,roberta/full/Vicomtech,0.76,0.745
2,roberta/training/Vicomtech,0.74,0.721
1,roberta/training/ISHate,0.66,0.626
4,roberta/documents/ISHate,0.64,0.625
7,roberta/full/ISHate,0.64,0.618
6,roberta/full/IHC,0.40,0.391
0,roberta/training/IHC,0.38,0.368
3,roberta/documents/IHC,0.38,0.368
